In [2]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.2 MB/s eta 0:00:0000:01


In [3]:
from ultralytics import YOLO
import torch, cv2
import numpy as np


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
detector = YOLO("yolov8n.pt")          
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").eval()
transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = transforms.small_transform


/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
Loading weights:  None


/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:00<00:00, 104MB/s] 
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [24]:
!wget https://ultralytics.com/images/bus.jpg
frame = cv2.imread("bus.jpg")




--2025-12-31 15:15:20--  https://ultralytics.com/images/bus.jpg
Resolving ultralytics.com (ultralytics.com)... 198.202.211.1
Connecting to ultralytics.com (ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ultralytics.com/images/bus.jpg [following]
--2025-12-31 15:15:20--  https://www.ultralytics.com/images/bus.jpg
Resolving www.ultralytics.com (www.ultralytics.com)... 198.202.211.1, 2620:cb:2000::1
Connecting to www.ultralytics.com (www.ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg [following]
--2025-12-31 15:15:20--  https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Loc

In [ ]:

TOMATO_ID = 0
POTATO_ID = 1

results = detector.track(
    source=frame,
    persist=True,
    tracker="bytetrack.yaml",
    classes=[TOMATO_ID, POTATO_ID]
)



0: 640x480 4 persons, 9.5ms
Speed: 3.9ms preprocess, 9.5ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 480)


In [ ]:
img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

input_batch = transform(img)   

with torch.no_grad():
    depth = midas(input_batch)

depth = torch.nn.functional.interpolate(
    depth.unsqueeze(1), size=frame.shape[:2], mode="bicubic", align_corners=False
).squeeze().cpu().numpy()


In [ ]:
fx, fy, cx, cy = 700, 700, frame.shape[1]//2, frame.shape[0]//2  

for r in results:
    for b in r.boxes:
        x1,y1,x2,y2 = map(int, b.xyxy[0])
        u, v = (x1+x2)//2, (y1+y2)//2

        Z = depth[v, u]             
        X = (u - cx) * Z / fx
        Y = (v - cy) * Z / fy

        track_id = int(b.id) if b.id is not None else -1


In [30]:
X,Y,Z

(np.float32(-341.08655), np.float32(155.95134), np.float32(638.3973))